In [ ]:
# Imports
import cv2
import numpy as np
import h5py
from pathlib import Path

# Load images from HDF5 file

In [ ]:
# Load images from HDF5 file
INPUT_HDF5 = r"path\to\your\input_dataset.h5"  # Change this to the path of your input dataset

# Load first 3 images from the first fold as examples
with h5py.File(INPUT_HDF5, 'r') as f:
    # Get first fold
    fold_name = 'fold_0'  # Adjust if needed
    images_dataset = f[fold_name]['image']
    
    # Load first 3 images
    imgs = []
    for i in range(min(3, len(images_dataset))):
        img = images_dataset[i]
        
        # Remove channel dimension if present (224, 224, 1) -> (224, 224)
        if len(img.shape) == 3 and img.shape[-1] == 1:
            img = img.squeeze(-1)
        
        # Convert to uint8 if needed
        if img.dtype != np.uint8:
            img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        
        # Convert grayscale to BGR for circle detection (HoughCircles expects BGR or grayscale)
        if len(img.shape) == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        
        imgs.append(img)

# Convert to grayscale for processing
gray_images = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in imgs]

# Rotation and Flipping for Best Match

In [ ]:
# Function to calculate Mean Squared Error between two images
def calculate_mse(imgA, imgB):
    # Images must be the same size
    err = np.sum((imgA.astype("float") - imgB.astype("float")) ** 2)
    err /= float(imgA.shape[0] * imgA.shape[1])
    return err

# Check all rotations and flips to find the best match
def get_best_match(image, prototype):
    # 1. Normalize: ensure same size and grayscale for pixel-wise comparison
    if image.shape != prototype.shape:
        image = cv2.resize(image, (prototype.shape[1], prototype.shape[0]))
    
    best_score = float("inf")
    best_version = None

    # Standard rotations: 0, 90, 180, 270 degrees
    rotation_codes = [None, cv2.ROTATE_90_CLOCKWISE, cv2.ROTATE_180, cv2.ROTATE_90_COUNTERCLOCKWISE]

    # Check 2 states: Original and Flipped (Horizontal)
    for flip in [False, True]:
        base_img = cv2.flip(image, 1) if flip else image
        
        for code in rotation_codes:
            # Apply rotation if it's not the 'None' (0 degrees) case
            candidate = cv2.rotate(base_img, code) if code is not None else base_img
            
            # 2. Similarity Metric: Mean Squared Error (Lower is better)
            # Use float64 to prevent overflow during subtraction
            diff = cv2.absdiff(candidate.astype("float"), prototype.astype("float"))
            score = np.mean(np.square(diff))
            
            if score < best_score:
                best_score = score
                best_version = candidate.copy() # Ensure we return a copy, not a reference
                
    return best_version, best_score

# Plot and show 3 examples

In [ ]:
import matplotlib.pyplot as plt
# Plot raw images
for i in range(len(gray_images)):
    plt.subplot(1, 3, i+1)
    plt.imshow(gray_images[i], cmap='gray')
    plt.title(f'Image {i+1}')
    plt.axis('off')
    plt.tight_layout()
plt.show()
for i in range(len(gray_images)):
    rot_flip = get_best_match(gray_images[i], gray_images[1])

    # Plot the best match in 1x3 array
    plt.subplot(1, 3, i+1)
    plt.imshow(rot_flip[0], cmap='gray')
    plt.title(f'Best Match, MSE: {rot_flip[1]:.2f}')
    plt.axis('off')
    plt.tight_layout()
plt.show()

# Rotate and or Flip an entire HDF5 file one fold at a time

In [ ]:
import h5py
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

def get_best_rotation(image, reference, allow_flip=True):
    """
    Find best rotation angle (0, 90, 180, 270) and flip state using MSE.
    
    Args:
        image: Image to align
        reference: Reference image to match
        allow_flip: If True, tests both flipped and non-flipped versions (8 total tests)
                   If False, only tests rotations without flipping (4 total tests)
    
    Returns (rotated_image, angle, flipped, score)
    """
    # Ensure same size
    if image.shape != reference.shape:
        image = cv2.resize(image, (reference.shape[1], reference.shape[0]))
    
    best_score = float("inf")
    best_image = None
    best_angle = 0
    best_flipped = False
    
    # Rotation codes: 0°, 90°, 180°, 270°
    rotation_map = {
        0: None,
        90: cv2.ROTATE_90_CLOCKWISE,
        180: cv2.ROTATE_180,
        270: cv2.ROTATE_90_COUNTERCLOCKWISE
    }
    
    # Try all combinations of rotation and flip
    flip_options = [False, True] if allow_flip else [False]
    
    for flip in flip_options:
        base_img = cv2.flip(image, 1) if flip else image
        
        for angle, code in rotation_map.items():
            # Apply rotation
            candidate = cv2.rotate(base_img, code) if code is not None else base_img
            
            # Calculate MSE
            diff = cv2.absdiff(candidate.astype("float"), reference.astype("float"))
            score = np.mean(np.square(diff))
            
            if score < best_score:
                best_score = score
                best_image = candidate.copy()
                best_angle = angle
                best_flipped = flip
    
    return best_image, best_angle, best_flipped, best_score

def process_hdf5_MSE(input_hdf5, output_hdf5, reference_fold, reference_idx, allow_flip=True):
    """
    Process entire HDF5 file using rotation matching based on MSE.
    
    Args:
        input_hdf5: Path to input HDF5 file
        output_hdf5: Path to output HDF5 file
        reference_fold: Fold containing reference image
        reference_idx: Index of reference image in fold
        allow_flip: If True, tests mirrored versions (8 tests per image)
                   If False, only tests rotations (4 tests per image)
    """
    
    print(f"Opening input HDF5: {input_hdf5}")
    print(f"Flip detection: {'ENABLED' if allow_flip else 'DISABLED'}")
    
    # Load reference image
    with h5py.File(input_hdf5, 'r') as f:
        ref_img = f[reference_fold]['image'][reference_idx]
        
        # Preprocess reference image
        if len(ref_img.shape) == 3 and ref_img.shape[-1] == 1:
            ref_img = ref_img.squeeze(-1)
        if ref_img.dtype != np.uint8:
            ref_img = cv2.normalize(ref_img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    print(f"Loaded reference image from {reference_fold}[{reference_idx}]")
    print(f"Reference shape: {ref_img.shape}")
    
    # Track statistics
    rotation_stats = {0: 0, 90: 0, 180: 0, 270: 0}
    flip_stats = {'flipped': 0, 'not_flipped': 0}
    
    # Open input and output HDF5 files
    with h5py.File(input_hdf5, 'r') as input_file:
        folds = sorted([key for key in input_file.keys() if key.startswith('fold_')])
        print(f"Found {len(folds)} folds: {folds}")
        
        with h5py.File(output_hdf5, 'w') as output_file:
            
            # Process each fold
            for fold_name in folds:
                fold_group = input_file[fold_name]
                output_group = output_file.create_group(fold_name)
                
                print(f"\n{'='*60}")
                print(f"Processing {fold_name}")
                print(f"{'='*60}")
                
                # Process each dataset in the fold
                for dataset_name in fold_group.keys():
                    input_dataset = fold_group[dataset_name]
                    
                    # If it's the images dataset, process it
                    if dataset_name == 'image' or (len(input_dataset.shape) >= 3 and input_dataset.shape[1] > 1):
                        num_images = len(input_dataset)
                        
                        # Create output dataset
                        output_dataset = output_group.create_dataset(
                            dataset_name,
                            shape=input_dataset.shape,
                            dtype=input_dataset.dtype
                        )
                        
                        # Copy metadata
                        for key, value in input_dataset.attrs.items():
                            output_dataset.attrs[key] = value
                        
                        # Process each image
                        for idx in tqdm(range(num_images), desc=f"{fold_name}/{dataset_name}", unit='img'):
                            
                            # Load image
                            img = input_dataset[idx]
                            
                            # Remove channel dimension if present
                            if len(img.shape) == 3 and img.shape[-1] == 1:
                                img = img.squeeze(-1)
                            
                            # Convert to uint8 if needed
                            if img.dtype != np.uint8:
                                img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
                                                       
                            # Find best rotation
                            corrected_image, angle, flipped, score = get_best_rotation(img, ref_img, allow_flip=allow_flip)
                            
                            # Update statistics
                            rotation_stats[angle] += 1
                            flip_stats['flipped' if flipped else 'not_flipped'] += 1
                            
                            # Debug: print first few images
                            if idx < 3:
                                print(f"Image {idx}: Angle={angle}°, Flipped={flipped}, MSE={score:.2f}")
                            
                            # Add channel dimension back if needed
                            if len(input_dataset.shape) == 4 and input_dataset.shape[-1] == 1:
                                corrected_image = np.expand_dims(corrected_image, axis=-1)
                            
                            # Save to output
                            output_dataset[idx] = corrected_image
                    
                    else:
                        # For non-image data (like labels), just copy
                        output_dataset = output_group.create_dataset(
                            dataset_name,
                            data=input_dataset[:]
                        )
                        # Copy metadata
                        for key, value in input_dataset.attrs.items():
                            output_dataset.attrs[key] = value
    
    # Print summary statistics
    print(f"\n{'='*60}")
    print(f"Successfully created: {output_hdf5}")
    print(f"{'='*60}")
    print(f"\nRotation Statistics:")
    for angle, count in sorted(rotation_stats.items()):
        print(f"  {angle}°: {count} images")
    print(f"\nFlip Statistics:")
    for state, count in flip_stats.items():
        print(f"  {state}: {count} images")
    
    return output_hdf5

def verify_hdf5_structure(hdf5_file):
    """Print structure of HDF5 file."""
    with h5py.File(hdf5_file, 'r') as f:
        print(f"\nHDF5 File: {hdf5_file}")
        print(f"{'='*60}")
        
        def print_structure(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  {name}: shape={obj.shape}, dtype={obj.dtype}")
            elif isinstance(obj, h5py.Group):
                print(f"{name}/ (Group)")
        
        f.visititems(print_structure)



In [ ]:
# Run the processing

if __name__ == "__main__":
    
    # Define paths
    INPUT_HDF5 = r"path\to\your\input_dataset.h5"       # Change this to the path of your input dataset for rotation and flipping
    OUTPUT_HDF5 = r"path\to\your\output_dataset.h5"     # Change this to desired output path for the processed dataset

    # Process the entire HDF5 file
    process_hdf5_MSE(
        INPUT_HDF5, 
        OUTPUT_HDF5,
        reference_fold='fold_0',
        reference_idx=6796,  # Adjust based on your reference image
        allow_flip=True  # Set to False to disable mirroring/flipping
    )
    
    # Verify output
    print("\n" + "="*60)
    print("Input file structure:")
    verify_hdf5_structure(INPUT_HDF5)
    
    print("\n" + "="*60)
    print("Output file structure:")
    verify_hdf5_structure(OUTPUT_HDF5)